# Aircraft Turnaround & Disruption Data Platform
### From public flight records to an auditable operations warehouse

This project reconstructs aircraft journeys and analyzes turnaround performance using real U.S. Bureau of Transportation Statistics flight records. It includes ingestion, contracts, immutable raw storage, Parquet, a SQL warehouse, incremental monthly loading, revision capture, tests, failure recovery and an interactive report.

**Quick start:** use a standard CPU runtime and select **Runtime → Run all**. The default downloads January–March 2025, approximately 79 MiB of ZIP files and 1.65 million flights. Processing takes several minutes and a few GB of disk. No API key is needed. The embedded `demo` mode contains 58,357 real flights over three days and runs faster.

The notebook is self-contained: it writes its reusable source modules from an embedded archive. Normal Run all execution requires package-installation access; live mode also requires the BTS download service. Source failures stay explicit. Google Drive is optional.

**What completion means:** the first run produces the warehouse, analytical outputs and interactive report for an already-observed historical period. There are no future flight outcomes to wait for.

## 1. My run settings
The default is the full historical quarter. A new month is a backfill/addition to the existing warehouse; changing `MONTHS` does not delete previously loaded months. `REFRESH_SOURCE` checks BTS for corrected files. Live and demo outputs are kept in different runtime folders.

In [ ]:
from pathlib import Path
import os
import sys

DATA_MODE = "live" #@param ["live", "demo"]
MONTHS = ["2025-01", "2025-02", "2025-03"]
REFRESH_SOURCE = False #@param {type:"boolean"}
MAX_TURN_MINUTES = 720 #@param {type:"integer"}
RUN_TESTS = True #@param {type:"boolean"}
VERIFY_FULL_REBUILD = False #@param {type:"boolean"}
USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
RESTORE_DRIVE_CHECKPOINT = True #@param {type:"boolean"}
AUTO_DOWNLOAD_RESULTS = True #@param {type:"boolean"}

# I keep the active database on local disk, even when I save a Drive checkpoint.
IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        pass
PROJECT_DIR = Path(os.environ.get("AIRCRAFT_PROJECT_DIR", "/content/aircraft_project" if IN_COLAB else "./aircraft_project")).resolve()
RUNTIME_DIR = PROJECT_DIR / ("runtime_" + DATA_MODE)
print("Mode:", DATA_MODE, "| Project:", PROJECT_DIR)

## 2. Restore the reusable project source
This cell writes Python modules, SQL models, tests, documentation and the real-data demo. It overwrites those packaged source files if rerun, so personal edits should be copied elsewhere first. Runtime data is not part of this embedded archive.

In [ ]:
#@title Restore packaged source and the real-data demo
import base64
import hashlib
import io
import zipfile

# This bundled source makes the notebook independent of a GitHub account.
SOURCE_ARCHIVE = "..."
source_bytes = base64.b64decode(SOURCE_ARCHIVE)
assert hashlib.sha256(source_bytes).hexdigest() == "864d051395d39600cfd68c64aaee929d67d1efe0ee1fa1e2a9cb282bf730d6db"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    for name in archive.namelist():
        target = (PROJECT_DIR / name).resolve()
        if not target.is_relative_to(PROJECT_DIR):
            raise ValueError("Unexpected source archive path")
    archive.extractall(PROJECT_DIR)
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Project source ready:", PROJECT_DIR)

## 3. Install the pinned environment
The runtime uses Python, pandas, Arrow, DuckDB and Plotly. The airport-reference and timezone database versions are pinned because time reconstruction should not vary silently between machines.

In [ ]:
import subprocess

# I install into this notebook's Python interpreter.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
print("Dependencies installed.")

## 4. Optional persistence
With Drive enabled, Colab asks for Google authorization. Checkpoint restoration is allowed only into an empty runtime. The large full checkpoint is saved at the end. Without Drive, the final results ZIP remains downloadable and the entire dataset can be rebuilt.

In [ ]:
from workspace_tools import safe_restore

drive_checkpoint = None
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("The Drive option is intended for Google Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
    drive_checkpoint = Path("/content/drive/MyDrive/Aircraft_Turnaround_Project") / (DATA_MODE + "_checkpoint.zip")
    if RESTORE_DRIVE_CHECKPOINT and drive_checkpoint.exists():
        if not RUNTIME_DIR.exists() or not any(RUNTIME_DIR.iterdir()):
            safe_restore(drive_checkpoint, RUNTIME_DIR)
            print("Saved runtime restored.")
        else:
            print("Using the existing local runtime.")
else:
    print("Using local runtime storage.")

## 5. Inspect the architecture and source contract
The reporting-carrier dataset avoids combining marketing codeshares into duplicate aircraft movements. Cancelled and diverted records stay in the warehouse. Turnaround links require clock-valid completed legs, matching endpoints, and gaps within the selected window.

| Layer | Stored output | Engineering purpose |
|---|---|---|
| Bronze | Original ZIP and source metadata | Reproducible source capture |
| Silver | Versioned validated Parquet; quarantine | Typed contracts, UTC times, rejected records |
| Warehouse | Current flights, versions, checkpoints, audit tables | Atomic partition replacement and historical provenance |
| Gold | Turnarounds, dimensions, SQL marts, HTML report | Reusable data products and coverage measures |

In [ ]:
import json
import importlib.metadata
import pandas as pd
from IPython.display import display, HTML, FileLink, Markdown
from aircraft_core import Config, connect, locked, sha256, write_json
from aircraft_models import metrics
from run_pipeline import run_pipeline, demo_inputs

active_months = ("2025-03",) if DATA_MODE == "demo" else tuple(MONTHS)
cfg = Config(RUNTIME_DIR, months=active_months, mode=DATA_MODE,
             refresh=REFRESH_SOURCE, max_turn_minutes=MAX_TURN_MINUTES)
source_inputs = demo_inputs() if DATA_MODE == "demo" else None
environment = {name: importlib.metadata.version(name) for name in ["pandas", "duckdb", "pyarrow", "airportsdata", "tzdata"]}
display(pd.DataFrame(environment.items(), columns=["Package", "Version"]))
print("Input partitions:", active_months)

## 6. Verify the domain and failure-recovery rules
The tests use isolated synthetic fixtures. They check daylight-saving changes, midnight, multi-day delays, cancellation barriers, cross-month links, duplicate keys, revisions, transaction rollback, and safe archive recovery. Synthetic fixtures never enter the real-data warehouse.

In [ ]:
if RUN_TESTS:
    test_run = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR, text=True, capture_output=True)
    print(test_run.stdout)
    if test_run.stderr:
        print(test_run.stderr)
    test_run.check_returncode()
else:
    print("Test execution was disabled in settings.")

## 7. Run the complete pipeline
The monthly load is the recovery boundary. If a later month fails, earlier committed months remain available. Rerunning reuses verified raw files and skips unchanged active partitions. Raw files and silver versions can survive failed transactions for diagnosis; the database checkpoint decides what is active.

In [ ]:
# Each partition reports its outcome as it finishes.
result = run_pipeline(cfg, inputs=source_inputs)
run_metrics = result["metrics"]
display(pd.DataFrame([
    ("Accepted flights", int(run_metrics["flights"])),
    ("Aircraft identifiers", int(run_metrics["aircraft"])),
    ("Airports", int(run_metrics["airports"])),
    ("Eligible turnarounds", int(run_metrics["eligible_turns"])),
    ("Cancellations", int(run_metrics["cancellations"])),
    ("Diversions", int(run_metrics["diversions"]))
], columns=["Measure", "Value"]))

## 8. Follow each partition back to its original file
Every source has a checksum, capture timestamp, reference hash and transformation version. An identical active source version should never insert another copy of a flight.

In [ ]:
def query(sql, parameters=None):
    connection = connect(cfg)
    try:
        return connection.execute(sql, parameters or []).fetchdf()
    finally:
        connection.close()

display(query("""SELECT c.partition_month, v.raw_rows, v.accepted_rows,
    v.rejected_rows, v.duplicate_rows, v.received_at, v.source_sha256, v.reference_hash
    FROM checkpoints c JOIN partition_versions v USING(version_id) ORDER BY c.partition_month"""))

## 9. Inspect the timestamp checks
Actual dates are reconstructed from reported elapsed time and signed delay, then checked against local clocks. This avoids the common mistake of treating a 01:00 departure after a long delay as occurring on the original service date. Ambiguous and nonexistent local times stay unresolved.

In [ ]:
display(query("SELECT time_status, count(*) AS flights FROM flights GROUP BY time_status ORDER BY flights DESC"))
display(query("SELECT count(*) AS missing_aircraft_identifier FROM flights WHERE tail_number IS NULL"))

## 10. Check aircraft-link coverage
The sequence includes cancelled, diverted and invalid legs before eligibility filtering. A missing or invalid leg must not disappear and create an unjustified connection between the flights around it. The first observed leg also has no known inbound movement.

In [ ]:
display(query("SELECT * FROM mart_link_coverage ORDER BY candidate_records DESC"))
display(pd.DataFrame(run_metrics["checks"].items(), columns=["Integrity check", "Violations"]))
assert all(value == 0 for value in run_metrics["checks"].values())

## 11. Compare airports with visible denominators
The conditional rate below asks: among eligible pairs whose inbound arrived at least 15 minutes late, what percentage also departed at least 15 minutes late? It does not establish the cause of the later delay. The minimum sample filter makes small groups visible as a deliberate analytical choice.

In [ ]:
MIN_LATE_INBOUND_PAIRS = 100
display(query("""SELECT airport, eligible_turns, late_inbound_turns,
    conditional_outbound_late_pct, median_actual_turn_min, median_recovery_min
    FROM mart_turnaround_airports WHERE late_inbound_turns >= ?
    ORDER BY conditional_outbound_late_pct DESC LIMIT 15""", [MIN_LATE_INBOUND_PAIRS]))

## 12. Plot the historical reliability timeline
Cancelled flights have a separate outcome; they are not assigned an arrival delay of zero. The arrival on-time denominator includes measured completed, non-diverted arrivals only.

In [ ]:
import plotly.express as px

daily = query("SELECT * FROM mart_daily_reliability ORDER BY flight_date")
figure = px.line(daily, x="flight_date", y="ontime_arrival_pct",
                 title="On-time arrival share across the loaded period",
                 labels={"flight_date": "Flight date", "ontime_arrival_pct": "On-time arrivals (%)"},
                 template="plotly_white")
figure.update_yaxes(range=[0, 100])
figure.show(renderer="colab" if IN_COLAB else "notebook")

## 13. Trace an aircraft journey
This example intentionally selects a disrupted aircraft day with multiple valid legs. It is useful for investigation, not a representative sample. Change the tail number and date to query any aircraft in the warehouse. The standalone report includes interactive selectors and a timeline.

In [ ]:
example = query("""SELECT tail_number, flight_date, count(*) AS legs, avg(dep_delay_min) AS mean_delay
    FROM flights WHERE time_valid AND tail_number IS NOT NULL GROUP BY tail_number, flight_date
    HAVING count(*) >= 5 ORDER BY mean_delay DESC, tail_number, flight_date LIMIT 1""").iloc[0]
TAIL_NUMBER = example["tail_number"]
FLIGHT_DATE = str(example["flight_date"].date())
print("Example:", TAIL_NUMBER, FLIGHT_DATE)
display(query("""SELECT f.carrier, f.flight_number, f.origin, f.dest, f.scheduled_dep_utc,
    f.actual_dep_utc, f.actual_arr_utc, f.dep_delay_min, f.arr_delay_min,
    r.link_status, r.planned_turn_min, r.actual_turn_min
    FROM flights f LEFT JOIN rotation_candidates r USING(flight_id)
    WHERE f.tail_number=? AND f.flight_date=? ORDER BY f.scheduled_dep_utc""", [TAIL_NUMBER, FLIGHT_DATE]))

## 14. Verify the turnaround arithmetic
For each eligible pair:

**actual turn − planned turn = outbound departure delay − inbound arrival delay.**

Positive `recovered_delay_min` means delay minutes decreased before the next departure. It does not imply that extra ground time caused the recovery.

In [ ]:
display(query("""SELECT airport, planned_turn_min, actual_turn_min, inbound_arr_delay_min,
    outbound_dep_delay_min, recovered_delay_min,
    (actual_turn_min-planned_turn_min)-added_delay_min AS identity_error_min
    FROM fact_turnarounds WHERE inbound_late ORDER BY recovered_delay_min DESC LIMIT 10"""))
display(query("SELECT * FROM mart_turn_bands ORDER BY planned_turn_band"))

## 15. Demonstrate an idempotent rerun
This reuses the captured source files and runs the same warehouse logic again. It should leave all business counts unchanged. Operational run logs legitimately gain another run.

In [ ]:
before = metrics(cfg)
cached_cfg = Config(RUNTIME_DIR, months=active_months, mode=DATA_MODE,
                    refresh=False, max_turn_minutes=MAX_TURN_MINUTES)
repeat = run_pipeline(cached_cfg, inputs=source_inputs, publish_output=False)
after = metrics(cfg)
assert before == after
assert all(part["status"] == "unchanged" for part in repeat["partitions"])
print("Rerun passed: business metrics unchanged and every input partition skipped its load.")

## 16. Independently rebuild from raw files
By default this exercises the real three-day demo in isolated directories to keep Run all manageable. Set `VERIFY_FULL_REBUILD=True` to compare the entire live cohort instead. The check compares every row of nine business tables in both directions; it then repeats replay to check idempotence. The main warehouse is not altered.

In [ ]:
import tempfile
from verify_rebuild import verify_rebuild

if VERIFY_FULL_REBUILD:
    verification_source = cached_cfg
else:
    lab_path = Path(tempfile.mkdtemp(prefix="aircraft_recovery_source_"))
    verification_source = Config(lab_path, months=("2025-03",), mode="demo")
    run_pipeline(verification_source, inputs=demo_inputs(), publish_output=False)
rebuild_path = Path(tempfile.mkdtemp(prefix="aircraft_rebuilt_"))
rebuild_evidence = verify_rebuild(verification_source, rebuild_path)
display(pd.DataFrame(rebuild_evidence["business_table_differences"].items(), columns=["Business table", "Differing rows"]))
write_json(RUNTIME_DIR / "logs/rebuild_verification.json", {
    "scope": "full_loaded_cohort" if VERIFY_FULL_REBUILD else "isolated_real_demo", **rebuild_evidence})

## 17. Open the interactive report
The report includes seven charts, airport and aircraft selectors, source checksums, exclusions, and definitions. It embeds its JavaScript, data and map outlines, so the downloaded HTML works without an external map service. The source links are ordinary optional hyperlinks.

In [ ]:
report_path = Path(result["report"])
display(FileLink(str(report_path)))
print("Interactive report:", report_path)
# I can change this to True for a large inline preview.
SHOW_INLINE_REPORT = False
if SHOW_INLINE_REPORT:
    display(HTML(report_path.read_text()))

## 18. Export the project and results
The results ZIP contains reusable source, SQL, tests, documentation, the real-data demo, current gold Parquet exports, the report and logs. The full raw archive and complete DuckDB file are retained in the runtime; the optional Drive checkpoint below includes them. The turnaround export alone has over a million rows in full mode, so this ZIP is larger than the notebook.

In [ ]:
from workspace_tools import export_project, make_checkpoint, copy_verified

with locked(cached_cfg):
    export_info = export_project(cached_cfg, result["directory"], PROJECT_DIR / "Aircraft_Turnaround_Run_Results.zip")
print("Results ZIP:", round(export_info["bytes"] / 1024**2, 1), "MiB")
display(FileLink(export_info["path"]))
if drive_checkpoint is not None:
    with locked(cached_cfg):
        checkpoint_info = make_checkpoint(RUNTIME_DIR, PROJECT_DIR / "Aircraft_Turnaround_Checkpoint.zip")
    copy_verified(checkpoint_info["path"], drive_checkpoint)
    print("Full checkpoint copied to Google Drive.")
if IN_COLAB and AUTO_DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(export_info["path"])

## 19. What I can explain in an interview

- Why reporting-carrier records are used and how the business key differs from aircraft identity.
- How signed delays and elapsed time resolve overnight dates, and where ambiguous clocks remain unresolved.
- Why invalid/cancelled/diverted records must remain in the sequence before link filtering.
- How atomic monthly replacement, immutable source versions, checksums and checkpoints make failures recoverable.
- Why a conditional late-departure rate is descriptive rather than causal.
- What the exact replay comparison proves, and what it cannot prove about source completeness.

The verified full-quarter build processes **1,645,503 flights** and reconstructs **1,432,291 eligible turns** at the default window. Compare these with your own results if sources or settings change. Suggested resume wording and architecture details are in `README.md`.

This is a complete historical batch platform. Continuous unattended ingestion needs an external scheduler; a Colab runtime is not a persistent service. Docker and CI configuration are included as deployment scaffolding, not as claims of deployment.

## Sources

- [BTS monthly reporting-carrier archives](https://transtats.bts.gov/PREZIP/)
- [BTS reporting definitions](https://www.bts.gov/explore-topics-and-geography/modes/aviation/number-40-technical-directive-reporting-time)
- [airportsdata reference and license](https://github.com/mborsetti/airportsdata)
- [Natural Earth public-domain map geometry](https://www.naturalearthdata.com/about/terms-of-use/)

`DATA_NOTICE.md` records the source scope, demo selection, licenses and analytical limits. The delivered `VERIFICATION.md` records what was actually tested.